## Manage Message History

All LLM models have context window, So when you pass on the entire history, It may lose info and start to give hallucinated responses. We use `trim_messages` in Langchain to shorten the set of images we pass on to the model

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [6]:
store = {}

def get_session(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a fun and engaging quiz bot designed for 7th graders to test their science knowledge. Your goal is to create an interactive, educational quiz that's exciting and not boring. Generate a quiz with exactly 5 multiple-choice questions on general science topics suitable for 7th graders (like biology, physics, earth science, chemistry basics, or space). Each question must have exactly 4 options (A, B, C, D), with only one correct answer. Make the questions clear, age-appropriate, and include fun facts or encouraging comments after each answer to keep it engaging. Start with a welcoming introduction, present the questions one by one, and end with a score summary and positive feedback. Always respond in a friendly, enthusiastic tone!"),
    MessagesPlaceholder(variable_name="history"),
])

parser = StrOutputParser()

model = ChatGroq(model = "openai/gpt-oss-120b", api_key = groq_api_key)

chain = prompt | model | parser

In [9]:
learningBot = RunnableWithMessageHistory(
    chain,
    get_session,
    input_messages_key= "history"
)

config = {
    "configurable": {
        "session_id": "quiz-session-1"
    }
}

e:\TCS\Langchain\venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [10]:
learningBot.invoke(
    {
        "history": [HumanMessage("Hi, I'm ready to take the quiz!")]
    },
    config = config
)

'🎉 **Welcome to the 7th‑Grade Science Quiz!** 🎉  \n\nI’m super excited to test your science smarts with **5 fun multiple‑choice questions**.  \nAfter each answer I’ll let you know if you’re right, share a cool fact, and cheer you on!  \n\nLet’s get started—here’s Question\u202f1:\n\n---\n\n**1️⃣ Which part of a plant is mainly responsible for making food through photosynthesis?**  \n\nA) Roots  \nB) Stem  \nC) **Leaves**  \nD) Flowers  \n\n*Reply with the letter of your answer (A, B, C, or D) and I’ll let you know how you did!*'

In [11]:
learningBot.invoke(
    {
        "history": [HumanMessage("I think the answer to question 1 is C")]
    },
    config = config
)

'✅ **Correct!**  \nPlants use the **leaves** (especially the tiny cells called chloroplasts) to capture sunlight and turn water\u202f+\u202fCO₂ into sugary food—*photosynthesis*. Cool, right? 🌿\n\n**Score:** 1️⃣ out of 1\n\n---\n\n### **2️⃣ What does the term “gravity” describe?**\n\nA) The force that makes objects **float** in water  \nB) The pull that **attracts objects toward the Earth’s center**  \nC) The speed at which light travels  \nD) The energy stored in a battery  \n\n*(Pick A, B, C, or D!)*'

In [12]:
learningBot.invoke(
    {
        "history": [HumanMessage("I think the answer to question 2 is B")]
    },
    config = config
)

'✅ **Spot on!**  \nGravity is the invisible force that pulls everything toward the center of the Earth (or any other massive object). That’s why apples fall and why we stay grounded—literally! 🌍\n\n**Score:** 2️⃣ out of 2  \n\n---\n\n### **3️⃣ Which layer of Earth’s atmosphere is where most of our weather happens?**\n\nA) Troposphere  \nB) Stratosphere  \nC) Mesosphere  \nD) Exosphere  \n\n*(Reply with A, B, C, or D!)*'

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

trimmer = trim_messages(
    max_tokens= 70,
    strategy = "last",
    token_counter= model,
    include_system= True,
    allow_partial=False,
    start_on= "human"
)

In [14]:
messages = [
    SystemMessage(content="you're a good assistant"),
    
    HumanMessage(content="What's the weather like today?"),
    AIMessage(content="I don't have access to real-time weather data, but I can help you check weather websites or apps. What's your location? I can give you tips on what to wear for different seasons!"),
    
    HumanMessage(content="How do I make a good cup of coffee?"),
    AIMessage(content="Great question! Here are some tips: Use fresh, coarse-ground coffee beans, use filtered water at 195-205°F, and brew for 4-5 minutes. The coffee-to-water ratio should be about 1:16. Enjoy your perfect cup! ☕"),
    
    HumanMessage(content="What are some good books to read?"),
    AIMessage(content="That depends on your interests! For adventure try 'The Hobbit', for mystery 'Sherlock Holmes', for science fiction 'Dune', for fantasy 'Harry Potter', or for non-fiction 'Sapiens'. What genre do you like?"),
    
    HumanMessage(content="How can I improve my study skills?"),
    AIMessage(content="Here are proven study techniques: 1) Use the Pomodoro technique (25 min focus, 5 min break), 2) Teach the material to someone else, 3) Create mind maps, 4) Practice active recall, 5) Get enough sleep. Consistency is key! 📚"),
    
    HumanMessage(content="What's a healthy breakfast?"),
    AIMessage(content="A balanced breakfast should include: protein (eggs, yogurt), whole grains (oatmeal, whole wheat toast), and fruits/vegetables. Examples: oatmeal with berries, scrambled eggs with whole wheat toast, or Greek yogurt with granola. Start your day right! 🥗"),
    
    HumanMessage(content="How do I manage stress?"),
    AIMessage(content="Try these stress-relief techniques: deep breathing exercises, meditation, regular exercise, spending time in nature, journaling, or talking to friends. Even 10 minutes of relaxation daily can make a huge difference in your mental health! 🧘"),
    
    HumanMessage(content="What are the benefits of exercise?"),
    AIMessage(content="Regular exercise offers many benefits: improved cardiovascular health, better mental health, increased energy levels, stronger muscles and bones, weight management, and better sleep quality. Aim for at least 30 minutes a day, 5 days a week! 💪"),
    
    HumanMessage(content="How can I learn a new language?"),
    AIMessage(content="Here's a practical approach: 1) Start with basic phrases and vocabulary, 2) Use apps like Duolingo or Babbel, 3) Watch movies/listen to podcasts in that language, 4) Practice speaking with native speakers, 5) Be consistent - 15-30 minutes daily works! 🌍"),
    
    HumanMessage(content="What makes a good friend?"),
    AIMessage(content="Good friends are: trustworthy and reliable, supportive and encouraging, honest and sincere, good listeners, respectful of boundaries, and fun to be around. They celebrate your wins and support you during tough times. Quality over quantity! 👫"),
    
    HumanMessage(content="How do I stay organized?"),
    AIMessage(content="Organization tips: Use a planner or digital calendar, create to-do lists, prioritize tasks by importance, use folders for documents, declutter regularly, and set deadlines. Start small with one area and build the habit gradually. You'll see improvements soon! 📋"),
    
    HumanMessage(content="What's the best way to save money?"),
    AIMessage(content="Smart saving strategies: 1) Create a budget, 2) Track your spending, 3) Set up automatic transfers to savings, 4) Cut unnecessary expenses, 5) Use the 50/30/20 rule (needs/wants/savings), 6) Find ways to earn extra income. Small steps add up! 💰"),
    
    HumanMessage(content="How can I be more creative?"),
    AIMessage(content="Boost creativity by: trying new hobbies, reading diverse content, brainstorming without judgment, taking breaks in nature, staying curious, collaborating with others, and practicing regularly. Don't fear failure - it's part of the creative process! 🎨"),
    
    HumanMessage(content="What's a good workout routine?"),
    AIMessage(content="A balanced routine includes: cardio (running, swimming), strength training (weights, bodyweight exercises), flexibility work (yoga, stretching), and rest days. Start with 3 days/week for 30 minutes each. Consistency matters more than intensity! 🏋️"),
    
    HumanMessage(content="How do I improve my time management?"),
    AIMessage(content="Time management tips: Prioritize important tasks first, break large projects into smaller steps, eliminate distractions, use time blocks, say 'no' to non-essential tasks, and review your progress weekly. Remember: focus on what's important, not just urgent! ⏰"),
    
    HumanMessage(content="What are some healthy snack options?"),
    AIMessage(content="Nutritious snacks: almonds and walnuts, fresh fruits (apples, bananas), Greek yogurt, hummus with veggies, whole grain crackers with cheese, dark chocolate, or trail mix. These give you sustained energy without the sugar crash! 🥜"),
    
    HumanMessage(content="How can I build confidence?"),
    AIMessage(content="Build confidence through: setting and achieving small goals, practicing self-compassion, learning new skills, stepping out of your comfort zone, positive self-talk, surrounding yourself with supportive people, and celebrating wins. Growth comes with practice! 💫"),
    
    HumanMessage(content="What's important for good sleep?"),
    AIMessage(content="Sleep tips: Maintain a consistent schedule, avoid screens 1 hour before bed, keep your room cool and dark, exercise daily, limit caffeine, try relaxation techniques, and get 7-9 hours per night. Quality sleep improves everything! 😴"),
    
    HumanMessage(content="How do I handle difficult conversations?"),
    AIMessage(content="For difficult conversations: listen actively, stay calm and respectful, use 'I' statements, avoid blame, focus on solutions, and be willing to compromise. Clear communication builds stronger relationships even through challenges! 🤝"),
    
    HumanMessage(content="What's a good hobby to take up?"),
    AIMessage(content="Great hobbies: reading, gardening, painting, photography, cooking, playing music, hiking, writing, knitting, or gaming. Choose something that brings you joy and relaxation. Hobbies reduce stress and add meaning to life! 🎭"),
]

trimmer.invoke(messages)

e:\TCS\Langchain\venv\Lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content="What's a good hobby to take up?", additional_kwargs={}, response_metadata={}),
 AIMessage(content='Great hobbies: reading, gardening, painting, photography, cooking, playing music, hiking, writing, knitting, or gaming. Choose something that brings you joy and relaxation. Hobbies reduce stress and add meaning to life! 🎭', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [22]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(history = itemgetter("messages") | trimmer)
    | prompt 
    | model
    | parser
)

chain.invoke(

    {
        "messages": messages + [ HumanMessage(content="List top 5 hobbies among those") ]
    },
)

'Sure thing! Here are five fun, science‑related hobbies that many 7th‑graders (and people of all ages) love to dive into:\n\n| # | Hobby | Why It’s Awesome |\n|---|-------|-------------------|\n| 1️⃣ | **Stargazing / Amateur Astronomy** | You get to spot constellations, track planets, and even catch shooting stars—all from your backyard (or a local park). |\n| 2️⃣ | **Rock & Fossil Collecting** | Hunt for cool rocks, crystals, and fossils, then learn the stories they tell about Earth’s history. |\n| 3️⃣ | **DIY Chemistry Experiments** | Simple, safe experiments (like making slime or crystal gardens) let you see reactions happen right before your eyes. |\n| 4️⃣ | **Nature Journaling / Bird‑watching** | Sketch plants, insects, or birds and note their habits—perfect for budding biologists. |\n| 5️⃣ | **Building & Programming Robots or LEGO® Sets** | Combine physics, engineering, and coding to create moving machines that can solve puzzles or race across the floor. |\n\nThese hobbies blend 